# PPE Object Detection FastAPI Demo

Run the existing FastAPI API on Google Colab and expose it with a temporary Cloudflare tunnel.

Before running: `Runtime > Change runtime type > T4 GPU`.

## Expected Google Drive Files

Copy these into `MyDrive/PPE_Object_Detection_Demo/` before running:

```text
app/
models/yolo26_ppe.onnx
models/model_metadata.json
requirements-gpu.txt
scripts/test_predict.py
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DEMO_DIR = Path('/content/drive/MyDrive/PPE_Object_Detection_Demo')
required_paths = [
    DEMO_DIR / 'app',
    DEMO_DIR / 'models' / 'yolo26_ppe.onnx',
    DEMO_DIR / 'models' / 'model_metadata.json',
    DEMO_DIR / 'requirements-gpu.txt',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required demo files:\n' + '\n'.join(missing))
print(f'Demo directory ready: {DEMO_DIR}')

In [ ]:
%cd /content/drive/MyDrive/PPE_Object_Detection_Demo
!python -m pip install --upgrade pip
!python -m pip install -r requirements-gpu.txt

In [ ]:
import os

os.environ['MODEL_PATH'] = 'models/yolo26_ppe.onnx'
os.environ['MODEL_METADATA_PATH'] = 'models/model_metadata.json'
os.environ['ENABLE_MODEL_WARMUP'] = 'true'
os.environ['ORT_INTRA_OP_THREADS'] = '1'
os.environ['ORT_INTER_OP_THREADS'] = '1'

import onnxruntime as ort
print('Available ONNX Runtime providers:', ort.get_available_providers())

In [ ]:
import subprocess
import time
import requests

api_process = subprocess.Popen([
    'python', '-m', 'uvicorn', 'app.main:app',
    '--host', '127.0.0.1',
    '--port', '8000',
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for attempt in range(60):
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=2)
        print(response.json())
        break
    except Exception:
        time.sleep(1)
else:
    api_process.terminate()
    raise RuntimeError('FastAPI did not start within 60 seconds')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import re
import subprocess

tunnel_process = subprocess.Popen([
    'cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for _ in range(120):
    line = tunnel_process.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    tunnel_process.terminate()
    raise RuntimeError('Could not create Cloudflare tunnel')

print('\nDemo API:', public_url)
print('Swagger UI:', public_url + '/docs')
print('Health:', public_url + '/health')

## Test Prediction From Colab

Upload an image here and send it to the public `/predict` endpoint.

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded.keys()))

with open(image_path, 'rb') as image_file:
    response = requests.post(
        public_url + '/predict',
        files={'file': (image_path, image_file, 'application/octet-stream')},
        data={'confidence_threshold': '0.25'},
        timeout=120,
    )

print('status_code:', response.status_code)
print(response.text)

## Stop Server

Run this when finished.

In [ ]:
tunnel_process.terminate()
api_process.terminate()
print('Stopped API and tunnel')